# [3장 2강] - Scaled Dot-Product Attention 계산 (1)

<aside>
🎯

**실습 목표**

- Scaled Dot-Product Attention을 함수 전체로 구현합니다.
- Padding Mask 적용 위치와 softmax 안정성을 확인합니다.
- Batch가 포함된 `[B, T, D]` 입력으로 로직을 확장합니다.
</aside>

---

## 핵심 실습. Scaled Dot-Product Attention 구현

### 시작 코드

```python
import torch

Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V = torch.tensor([[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]])
```

### 수행해야 할 작업

1. `scaled_dot_product_attention(Q, K, V)` 함수를 작성하세요.
2. score를 `sqrt(d_k)`로 나누고 마지막 차원에 softmax를 적용하세요.
3. output과 attention weight를 함께 반환하세요.
4. Shape와 각 weight 행의 합을 검증하세요.

    
  **해설**
    
  Softmax는 Key 방향인 마지막 차원에 적용합니다. 각 Query가 모든 Key에 배분한 weight의 합이 1이 되고, 이 weight로 V를 가중합해 context vector를 만듭니다.

In [1]:
import math
import torch

Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])


def scaled_dot_product_attention(Q, K, V):
    if Q.size(-1) != K.size(-1):
        raise ValueError("Q와 K의 마지막 차원이 다릅니다.")
    if K.size(-2) != V.size(-2):
        raise ValueError("K와 V의 token 수가 다릅니다.")

    # 큰 d_k에서 dot product가 커지는 현상을 줄이기 위해 scaling합니다.
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


output, weights = scaled_dot_product_attention(Q, K, V)
print("weights:\n", weights)
print("output:\n", output)

assert weights.shape == (3, 3)
assert output.shape == (3, 2)
assert torch.allclose(weights.sum(dim=-1), torch.ones(3))

weights:
 tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
output:
 tensor([[6.0167, 3.9833],
        [3.9833, 6.0167],
        [5.0000, 5.0000]])


## 핵심 보조 실습. Padding Mask가 포함된 Attention 작성

### 시작 코드

```python
Q = K = V = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [9.0, 9.0],  # padding 위치라고 가정
])
key_mask = torch.tensor([1, 1, 1, 0], dtype=torch.bool)
```

### 수행해야 할 작업

1. `masked_attention(Q, K, V, key_mask)` 함수를 작성하세요.
2. Padding Key 열의 score를 softmax 전에 매우 작은 값으로 바꾸세요.
3. Padding 열의 weight가 0에 가까운지 검증하세요.
4. 모든 위치가 mask인 입력은 명시적으로 오류 처리하세요.

    
  **해설**
  
   Mask는 softmax 이후 weight를 0으로 곱하는 것보다 softmax 이전 score에 적용하는 편이 안전합니다. 이후에 단순히 0을 곱하면 남은 위치의 weight 합이 1이 아니게 됩니다.

In [2]:
import math
import torch

Q = K = V = torch.tensor([[1., 0.], [0., 1.], [1., 1.], [9., 9.]])
key_mask = torch.tensor([1, 1, 1, 0], dtype=torch.bool)


def masked_attention(Q, K, V, key_mask):
    if key_mask.ndim != 1 or key_mask.numel() != K.size(-2):
        raise ValueError("key_mask shape가 Key 길이와 맞지 않습니다.")
    if not key_mask.any():
        raise ValueError("모든 Key가 mask되어 있습니다.")

    scores = (Q @ K.T) / math.sqrt(Q.size(-1))
    # 열 방향으로 broadcast해 padding Key를 어떤 Query도 보지 못하게 합니다.
    masked_scores = scores.masked_fill(~key_mask.unsqueeze(0), float("-inf"))
    weights = torch.softmax(masked_scores, dim=-1)
    output = weights @ V
    return output, weights


output, weights = masked_attention(Q, K, V, key_mask)
print(weights)
assert torch.allclose(weights[:, -1], torch.zeros(4))
assert torch.allclose(weights.sum(dim=-1), torch.ones(4))

tensor([[0.4011, 0.1978, 0.4011, 0.0000],
        [0.1978, 0.4011, 0.4011, 0.0000],
        [0.2483, 0.2483, 0.5035, 0.0000],
        [0.0017, 0.0017, 0.9966, 0.0000]])


## 참고·심화 실습. Batch Attention으로 확장

### 시작 코드

```python
import torch

torch.manual_seed(42)
Q = torch.randn(2, 4, 6)
K = torch.randn(2, 4, 6)
V = torch.randn(2, 4, 6)
attention_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)
```

### 수행해야 할 작업

1. `[B, T, D]`를 처리하는 `batch_attention` 함수를 작성하세요.
2. `torch.matmul`과 `transpose(-2, -1)`을 사용하세요.
3. Mask를 `[B, 1, T]`로 바꾸어 Key 열에 적용하세요.
4. output `[B, T, D]`, weights `[B, T, T]`를 검증하세요.

    
  **해설**
    
   Batch 차원은 attention 계산에 섞이지 않고 각 sample별로 유지됩니다. `transpose(-2, -1)`를 사용하면 batch가 추가되어도 마지막 두 차원만 바뀌어 재사용하기 쉽습니다.

In [6]:
import math
import torch

torch.manual_seed(42)
Q = torch.randn(2, 4, 6)
K = torch.randn(2, 4, 6)
V = torch.randn(2, 4, 6)
attention_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)


def batch_attention(Q, K, V, attention_mask):
    if Q.ndim != 3 or K.ndim != 3 or V.ndim != 3:
        raise ValueError("Q, K, V는 [B, T, D]여야 합니다.")

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    key_mask = attention_mask.unsqueeze(1)
    scores = scores.masked_fill(~key_mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights


output, weights = batch_attention(Q, K, V, attention_mask)
print("scores shape:", tuple(weights.shape))
print("output shape:", tuple(output.shape))

assert weights.shape == (2, 4, 4)
assert output.shape == (2, 4, 6)
assert torch.allclose(weights[1, :, 2:], torch.zeros(4, 2))

scores shape: (2, 4, 4)
output shape: (2, 4, 6)
